# 097 — Datos sintéticos: utilidad y contaminación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** (a) P(0 raros) = (1 − 0.05)²⁰ = 0.95²⁰ ≈ **0.358**. (b) Sobrevivir una
generación tiene probabilidad ≈ 1 − 0.358 = 0.642; tres generaciones ≈ 0.642³ ≈ **0.265**.
La aproximación supone p̂ ≈ 0.05 mientras la cola viva; en realidad p̂ fluctúa (0, 0.05,
0.10…), pero el mensaje es idéntico: p̂ = 0 es absorbente y domina a largo plazo.

**Ejercicio 2.** La trayectoria depende de la semilla porque el colapso es un fenómeno
de muestreo, no del modelo: con una semilla la cola puede sobrevivir (incluso subir por
azar), con otra desaparece en la primera generación. Ninguna corrida individual
"demuestra" el colapso; lo demuestra la distribución sobre corridas (ejercicio 1).

**Ejercicio 3.** TSTR = 0.72 / 0.90 = **0.80** < 0.9 → no pasa el umbral: los datos
sintéticos no preservan suficiente relación feature–etiqueta. TSTR no dice nada sobre
privacidad: un generador que memoriza y copia registros reales tendría TSTR ≈ 1 y
privacidad nula. Utilidad y privacidad se miden por separado.

**Ejercicio 4.** El contrato JSON expone `kind` y `evidence`; solo la evidencia
inspeccionable autoriza conclusiones.

In [ ]:
result = run_lab("evaluation", seed=97)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica de los ejercicios
import random

# Ejercicio 1
p, N = 0.05, 20
p_ext = (1 - p) ** N
p_surv3 = (1 - p_ext) ** 3
print(f"P(0 raros en una generación) = {p_ext:.3f}")
print(f"P(cola viva tras 3 generaciones) ≈ {p_surv3:.3f}")

# Ejercicio 2: tres generaciones de muestreo recursivo
def simular(seed, p0=0.05, N=20, gens=3):
    random.seed(seed)
    p_hat, tray = p0, [p0]
    for _ in range(gens):
        raros = sum(1 for _ in range(N) if random.random() < p_hat)
        p_hat = raros / N
        tray.append(p_hat)
    return tray

for s in (94, 7):
    print(f"seed={s}: trayectoria de p̂ = {simular(s)}")

# Ejercicio 3
tstr = 0.72 / 0.90
print(f"TSTR = {tstr:.3f}  →  {'pasa' if tstr >= 0.9 else 'NO pasa'} el umbral 0.9")
assert abs(tstr - 0.8) < 1e-9

## Reflexión

1. Con p_rara = 0.05 y N = 20, el riesgo de perder la cola en una generación es 0.95²⁰ ≈ 0.36. ¿Cuánto vale con N = 200 y qué implica sobre el tamaño de muestra en pipelines recursivos?
2. ¿Por qué TSTR mide utilidad mejor que una métrica de fidelidad marginal (p. ej. parecido visual), y qué relación captura que la fidelidad no captura?
3. Si en un crawl web no puedes distinguir el contenido generado, ¿qué aporta la procedencia activa (clase 098) como mitigación del collapse y cuál es su límite (¿quién marca y quién no)?